# QLoRA-train Qwen3-8B on macro traces (Kaggle free GPU)

Thin wrapper around `train/train_qlora.py` from
[Fine-Tune-Models](https://github.com/ashcastelinocs124/Fine-Tune-Models).

**Before running:**
1. **Settings → Accelerator**: `GPU T4 x2` or `GPU P100` (script uses one GPU).
2. **Settings → Internet**: ON (clones the repo, downloads Qwen3-8B).
3. **Add Input → your private Kaggle Dataset** containing the clean traces
   (`data/traces/clean/*.json` from the repo pipeline). Set its name in the
   config cell below.

Output: LoRA adapter zipped at `/kaggle/working/qwen3-8b-macro-lora.zip` —
download it from the Output tab.

In [ ]:
# ---- Config ----
TRACES_DATASET = "macro-traces-clean"  # name of the attached Kaggle Dataset
MAX_LEN = 8192   # 16384 OOMs on 16GB T4/P100 with an 8B model; raise on bigger GPUs
EPOCHS = 2
LR = 2e-4

In [ ]:
%cd /kaggle/working
!rm -rf Fine-Tune-Models
!git clone --depth 1 https://github.com/ashcastelinocs124/Fine-Tune-Models.git
%cd Fine-Tune-Models
# pyproject pins transformers<5; peft/bitsandbytes/accelerate are the GPU extras
!pip install -q -e . peft bitsandbytes accelerate

In [ ]:
# ---- Stage traces from the attached Kaggle Dataset ----
import glob, os, shutil

os.makedirs("data/traces/clean", exist_ok=True)
src = f"/kaggle/input/{TRACES_DATASET}"
files = glob.glob(f"{src}/**/*.json", recursive=True)
assert files, f"no .json traces under {src} — attach your traces dataset and check TRACES_DATASET"
for f in files:
    shutil.copy(f, "data/traces/clean/")
print(f"staged {len(files)} traces")

In [ ]:
!python train/train_qlora.py --clean data/traces/clean --out outputs/qwen3-8b-macro-lora \
    --epochs {EPOCHS} --lr {LR} --max-len {MAX_LEN}

In [ ]:
# ---- Package the adapter for download ----
import shutil

shutil.make_archive("/kaggle/working/qwen3-8b-macro-lora", "zip", "outputs/qwen3-8b-macro-lora")
print("adapter at /kaggle/working/qwen3-8b-macro-lora.zip")